In [0]:
%pip install faker pandas

import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta
import os

# Caminho do Volume criado no Databricks
volume_path = "/Volumes/workspace/default/raw_data"
os.makedirs(volume_path, exist_ok=True)

fake = Faker('pt_BR')

def generate_data(num_clientes=50):
    clientes, contas, cartoes = [], [], []
    
    # GERANDO CLIENTES
    for id_cliente in range(1, num_clientes + 1):
        segmento = random.choice(["Varejo", "Uniclass", "Personnalite", "Zenith"])
        clientes.append({
            "id_cliente": id_cliente,
            "cpf": fake.cpf(),
            "nome": fake.name(),
            "cidade": fake.city(),
            "estado": fake.estado_sigla(),
            "renda": round(random.uniform(2000.0, 45000.0), 2),
            "segmento": segmento,
            "data_atualizacao": (datetime.now() - timedelta(days=random.randint(50, 100))).strftime("%Y-%m-%d %H:%M:%S"),
            "operacao": "I"
        })
        
        # GERANDO CONTAS
        for _ in range(random.randint(1, 2)):
            id_conta = fake.unique.random_int(min=1000, max=9999)
            contas.append({
                "id_conta": id_conta,
                "id_cliente": id_cliente,
                "tipo_conta": random.choice(["Corrente", "Poupanca", "Pagamento"]),
                "status_conta": "Ativa",
                "data_abertura": (datetime.now() - timedelta(days=random.randint(100, 365))).strftime("%Y-%m-%d"),
                "data_atualizacao": (datetime.now() - timedelta(days=random.randint(50, 100))).strftime("%Y-%m-%d %H:%M:%S"),
                "operacao": "I"
            })
            
            # GERANDO CARTÕES
            id_cartao = fake.unique.random_int(min=10000, max=99999)
            cartoes.append({
                "id_cartao": id_cartao,
                "id_conta": id_conta,
                "tipo_cartao": random.choice(["Credito", "Debito", "Multiplo"]),
                "limite": round(random.uniform(1000.0, 50000.0), 2),
                "status_cartao": "Ativo",
                "data_atualizacao": (datetime.now() - timedelta(days=random.randint(50, 100))).strftime("%Y-%m-%d %H:%M:%S"),
                "operacao": "I"
            })

    df_clientes = pd.DataFrame(clientes)
    df_contas = pd.DataFrame(contas)
    df_cartoes = pd.DataFrame(cartoes)

    # --- INJETANDO PROBLEMAS SIMULADOS OBRIGATÓRIOS ---
    
    # 1. SCD Tipo 2: Atualização de Cliente 
    cliente_update = df_clientes.iloc[0].copy()
    cliente_update['renda'] = 65000.00
    cliente_update['segmento'] = "Zenith"
    cliente_update['data_atualizacao'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cliente_update['operacao'] = "U"
    
    # 2. Qualidade: Dado inválido 
    cliente_invalido = df_clientes.iloc[1].copy()
    cliente_invalido['cpf'] = None
    cliente_invalido['operacao'] = "I"

    # 3. Status Alterado: Cartão Cancelado
    cartao_cancelado = df_cartoes.iloc[0].copy()
    cartao_cancelado['status_cartao'] = "Cancelado"
    cartao_cancelado['data_atualizacao'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cartao_cancelado['operacao'] = "U"

    df_clientes = pd.concat([df_clientes, pd.DataFrame([cliente_update, cliente_invalido])]).sample(frac=1).reset_index(drop=True)
    df_cartoes = pd.concat([df_cartoes, pd.DataFrame([cartao_cancelado])]).sample(frac=1).reset_index(drop=True)

    # Salvando diretamente no Volume do Unity Catalog
    df_clientes.to_csv(f"{volume_path}/clientes_cdc.csv", index=False)
    df_contas.to_csv(f"{volume_path}/contas_cdc.csv", index=False)
    df_cartoes.to_csv(f"{volume_path}/cartoes_cdc.csv", index=False)
    
    print(f"Arquivos gerados com sucesso no caminho: {volume_path}")

generate_data()

In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta
import uuid

volume_path = "/Volumes/workspace/default/raw_data"

def generate_transactions():
    print("Gerando transações, eventos e estornos...")
    
    # Pega os IDs de cartões válidos que geramos anteriormente
    try:
        df_cartoes = pd.read_csv(f"{volume_path}/cartoes_cdc.csv")
        cartoes_ids = df_cartoes['id_cartao'].tolist()
    except:
        print("Erro: Não foi possível ler cartoes_cdc.csv. Usando IDs mockados temporários.")
        cartoes_ids = [10001, 10002, 10003, 10004, 10005]

    transacoes, eventos, estornos = [], [], []
    
    # GERANDO TRANSAÇÕES
    for _ in range(200):
        id_transacao = str(uuid.uuid4())
        data_trans = datetime.now() - timedelta(days=random.randint(1, 30), hours=random.randint(1, 23))
        
        transacoes.append({
            "id_transacao": id_transacao,
            "id_cartao": random.choice(cartoes_ids),
            "data_transacao": data_trans.strftime("%Y-%m-%d %H:%M:%S"),
            "valor": round(random.uniform(10.0, 5000.0), 2),
            "mcc": random.choice(["5411", "5812", "5999", "4511"]), # Supermercado, Restaurante, Varejo, Cia Aérea
            "estabelecimento": f"Estabelecimento {random.randint(1, 50)}",
            "canal": random.choice(["Fisico", "Online"]),
            "pais": "BR",
            "moeda": "BRL"
        })
        
        # GERANDO EVENTOS DE RISCO (5% de chance)
        if random.random() < 0.05:
            eventos.append({
                "id_evento": str(uuid.uuid4()),
                "id_transacao": id_transacao,
                "tipo_evento": random.choice(["Fraude", "Suspeita", "Inconsistencia"]),
                "severidade": random.choice(["Alta", "Media"]),
                "data_evento": (data_trans + timedelta(hours=2)).strftime("%Y-%m-%d %H:%M:%S")
            })
            
        # GERANDO ESTORNOS (5% de chance)
        if random.random() < 0.05:
            estornos.append({
                "id_estorno": str(uuid.uuid4()),
                "id_transacao": id_transacao,
                "data_estorno": (data_trans + timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S"),
                "motivo": random.choice(["Chargeback", "Desacordo Comercial", "Devolucao"])
            })

    df_transacoes = pd.DataFrame(transacoes)
    
    # INJETANDO PROBLEMAS OBRIGATÓRIOS
    # 1. Duplicidade (Mesmo id_transacao aparecendo duas vezes)
    transacao_duplicada = df_transacoes.iloc[0].copy()
    
    # 2. Dados Atrasados (Data muito antiga)
    transacao_atrasada = df_transacoes.iloc[1].copy()
    transacao_atrasada['data_transacao'] = (datetime.now() - timedelta(days=150)).strftime("%Y-%m-%d %H:%M:%S")
    
    df_transacoes = pd.concat([df_transacoes, pd.DataFrame([transacao_duplicada, transacao_atrasada])]).sample(frac=1).reset_index(drop=True)

    # Salvando no Volume
    df_transacoes.to_csv(f"{volume_path}/transacoes.csv", index=False)
    pd.DataFrame(eventos).to_csv(f"{volume_path}/eventos_risco.csv", index=False)
    pd.DataFrame(estornos).to_csv(f"{volume_path}/estornos.csv", index=False)
    
    print(f"Arquivos transacoes.csv, eventos_risco.csv e estornos.csv gerados no volume {volume_path}!")

generate_transactions()